# Buffered Loading with Shuffle Buffer — Solution

In [ ]:
import torch
import random
from torch.utils.data import IterableDataset, DataLoader

class SequentialDataset(IterableDataset):
    def __init__(self, N):
        self.N = N
    def __iter__(self):
        for i in range(self.N):
            yield torch.tensor(i, dtype=torch.long), torch.tensor(float(i))

In [ ]:
class ShuffleBufferDataset(IterableDataset):
    """Wraps an IterableDataset and shuffles its output via a fixed-size buffer."""

    def __init__(self, base_dataset: IterableDataset, buffer_size: int):
        super().__init__()
        self.base_dataset = base_dataset
        self.buffer_size = buffer_size

    def __iter__(self):
        it = iter(self.base_dataset)

        # Fill the buffer
        buffer = []
        for _ in range(self.buffer_size):
            try:
                buffer.append(next(it))
            except StopIteration:
                break

        # Yield random elements, replacing with new ones from the iterator
        for item in it:
            idx = random.randint(0, len(buffer) - 1)
            yield buffer[idx]
            buffer[idx] = item

        # Drain the remaining buffer in random order
        random.shuffle(buffer)
        for item in buffer:
            yield item


# Demo
shuffled = ShuffleBufferDataset(SequentialDataset(20), buffer_size=5)
print("Shuffled order:", [idx.item() for idx, _ in shuffled])